Setup Clearml

In [2]:
%env CLEARML_WEB_HOST=https://app.5ccsagap.er.kcl.ac.uk/
%env CLEARML_API_HOST=https://api.5ccsagap.er.kcl.ac.uk
%env CLEARML_FILES_HOST=https://files.5ccsagap.er.kcl.ac.uk
# GAP_group3_embedding
%env CLEARML_API_ACCESS_KEY=F3JFSR84AVXM6OTRRRUBQ5QPVHMTWV
%env CLEARML_API_SECRET_KEY=tda_99OfnqWYSguFYS5wH6FtKuVBoryYeREQVh2MPdcXct0hGiNEnfxeIOBpXPH1fcE

env: CLEARML_WEB_HOST=https://app.5ccsagap.er.kcl.ac.uk/
env: CLEARML_API_HOST=https://api.5ccsagap.er.kcl.ac.uk
env: CLEARML_FILES_HOST=https://files.5ccsagap.er.kcl.ac.uk
env: CLEARML_API_ACCESS_KEY=F3JFSR84AVXM6OTRRRUBQ5QPVHMTWV
env: CLEARML_API_SECRET_KEY=tda_99OfnqWYSguFYS5wH6FtKuVBoryYeREQVh2MPdcXct0hGiNEnfxeIOBpXPH1fcE


Load Embedding Model

In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os
 
load_dotenv()
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def openai_small(text):
    response = client.embeddings.create(
        input=[text],
        model="text-embedding-3-small"
    )
    return response.data[0].embedding  

def openai_large(text):
    response = client.embeddings.create(
        input=[text],
        model="text-embedding-3-large"
    )
    return response.data[0].embedding   


In [4]:
from sentence_transformers import SentenceTransformer
 
hf_mini_model = SentenceTransformer("all-MiniLM-L6-v2")   
hf_mpnet_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

def hf_mini(text):
    return hf_mini_model.encode(text).tolist()

def hf_mpnet(text):
    return hf_mpnet_model.encode(text).tolist()

c:\Users\guoji\Group-3\gjwenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Define cosine similarity

In [5]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def cosine_sim(vec1, vec2):
    vec1 = np.array(vec1).reshape(1, -1)
    vec2 = np.array(vec2).reshape(1, -1)
    return cosine_similarity(vec1, vec2)[0][0]

Semantic Textual Similarity

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ppxscal/academic_embeddings_test")
 


c:\Users\guoji\Group-3\gjwenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 120639/120639 [00:01<00:00, 86083.76 examples/s]


KeyError: "Invalid key: 0. Please first select a split. For example: `my_dataset_dictionary['train'][0]`. Available splits: ['train']"

In [15]:
for i in dataset['train']:
    if len(i['Query Text'])<=50 and i['Query Text'][-1]=='?':
        print(i['Query Text']) 

Amazon mechanical turk: Gold mine or coal mine?
Amazon mechanical turk: Gold mine or coal mine?
Amazon mechanical turk: Gold mine or coal mine?
Amazon mechanical turk: Gold mine or coal mine?
Amazon mechanical turk: Gold mine or coal mine?
Amazon mechanical turk: Gold mine or coal mine?
Amazon mechanical turk: Gold mine or coal mine?
Amazon mechanical turk: Gold mine or coal mine?


In [7]:
import random
# randomly sample 50 from the dataset
random.seed(42)   
small_benchmark = dataset.shuffle(seed=42).select(range(50))


In [ ]:
def evaluate_on_mteb(dataset, embed_function):
    similarities = []
    human_scores = []

    for example in dataset:
        sent1, sent2, score = example["sentence1"], example["sentence2"], example["similarity_score"]
        
        emb1 = embed_function(sent1)
        emb2 = embed_function(sent2)

        model_similarity = cosine_sim(emb1, emb2)

        similarities.append(model_similarity)
        human_scores.append(score / 5.0)   

    from scipy.stats import pearsonr
    correlation, _ = pearsonr(similarities, human_scores)
    
    return correlation
 
hf_mini_correlation = evaluate_on_mteb(small_benchmark, hf_mini)
hf_mpnet_correlation = evaluate_on_mteb(small_benchmark, hf_mpnet) 
openai_small_correlation = evaluate_on_mteb(small_benchmark, openai_small)
openai_large_correlation = evaluate_on_mteb(small_benchmark, openai_large)

print(f"OpenAI Small Model Pearson Correlation: {openai_small_correlation:.4f}")
print(f"OpenAI Large Model Pearson Correlation: {openai_large_correlation:.4f}")
print(f"hf_mini_correlation Pearson Correlation: {hf_mini_correlation:.4f}")
print(f"hf_mpnet_correlation Pearson Correlation: {hf_mpnet_correlation:.4f}")

In [ ]:
# results from mteb 
# openai_small_correlation = 0.8673
# openai_large_correlation = 0.8668
# hf_mini_correlation = 0.7993
# hf_mpnet_correlation = 0.8423

In [10]:
from clearml import Task 
task = Task.init(project_name="Embedding model evaluation", task_name="Compare Embedding Models - Semantic Similarity2")
 
task.get_logger().report_scalar("Semantic Meaning Pearson Correlation", "OpenAI Small Model", openai_small_correlation, iteration=1)
task.get_logger().report_scalar("Semantic Meaning Pearson Correlation", "OpenAI Large Model", openai_large_correlation, iteration=1)
task.get_logger().report_scalar("Semantic Meaning Pearson Correlation", "HF mini Model", hf_mini_correlation, iteration=1) 
task.get_logger().report_scalar("Semantic Meaning Pearson Correlation", "HF mpnet Model", hf_mpnet_correlation, iteration=1)

task.close()


ClearML Task: overwriting (reusing) task id=2bf8ddb537014e02b871faaf4d355c8b
ClearML results page: https://app.5ccsagap.er.kcl.ac.uk/projects/8f37d229facb4d049719ecba5f506163/experiments/2bf8ddb537014e02b871faaf4d355c8b/output/log
2025-03-11 20:59:03,187 - clearml.Task - INFO - Waiting for repository detection and full package requirement analysis
2025-03-11 20:59:03,627 - clearml.Task - INFO - Finished repository detection and package analysis


Retrieval Evaluation

In [ ]:
# import random 
# from beir import util, LoggingHandler
# from beir.datasets.data_loader import GenericDataLoader
# from beir.retrieval.evaluation import EvaluateRetrieval
# from sentence_transformers import SentenceTransformer 

# # download MSMARCO dataset
# dataset = "msmarco"
# data_path = f"datasets/{dataset}"   

# ##uncomment to download the dataset for testing
# # url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

# # # download and unzip
# # util.download_and_unzip(url, data_path)
 
# data_loader = GenericDataLoader(data_path)
# corpus, queries, qrels = data_loader.load(split="test")

# # sample 50 instances only
# num_samples = 50   
# selected_query_ids = random.sample(list(queries.keys()), min(num_samples, len(queries)))

# msmarco_subset = []
# for query_id in selected_query_ids:
#     query_text = queries[query_id]
#     if query_id in qrels:  # only handle queries that have ground truth
#         relevant_docs = [doc_id for doc_id in qrels[query_id]]  # ground truth document id
#         candidate_doc_ids = relevant_docs + random.sample(list(corpus.keys()), 10)  # 10 random incorrect options
#         candidate_docs = [corpus[doc_id]["text"] for doc_id in candidate_doc_ids]  
#         msmarco_subset.append({
#             "query": query_text,
#             "relevant_docs": relevant_docs,
#             "candidate_docs": candidate_docs
#         })
 


CLEARML-SERVER new package available: UPGRADE to v2.0.0 is recommended!
Release Notes:
### Breaking Changes

MongoDB major version was upgraded from v5.x to 6.x.
Please note that if your current ClearML Server version is smaller than v1.17 (where MongoDB v5.x was first used), you'll need to first upgrade to ClearML Server v1.17.
#### Upgrading to ClearML Server v1.17 from a previous version
- If using docker-compose,  use the following docker-compose files:
  * [docker-compose file](https://github.com/allegroai/clearml-server/blob/2976ce69cc91550a3614996e8a8d8cd799af2efd/upgrade/1_17_to_2_0/docker-compose.yml)
  * [docker-compose file foe Windows](https://github.com/allegroai/clearml-server/blob/2976ce69cc91550a3614996e8a8d8cd799af2efd/upgrade/1_17_to_2_0/docker-compose-win10.yml)

### New Features

- New look and feel: Full light/dark themes ([clearml #1297](https://github.com/allegroai/clearml/issues/1297))
- New UI task creation options
  - Support bash as well as python scripts
  -

100%|██████████| 8841823/8841823 [00:29<00:00, 302863.12it/s]


In [12]:
def evaluate_retrieval(dataset, embed_function):
    retrieved_ranks = []
    for example in dataset:
        query = example["query"]
        relevant_docs = example["relevant_docs"]
        candidate_docs = example["candidate_docs"]

        query_embedding = embed_function(query)
        candidate_embeddings = [embed_function(doc) for doc in candidate_docs]

        similarities = [cosine_sim(query_embedding, emb) for emb in candidate_embeddings]
        ranked_docs = sorted(zip(candidate_docs, similarities), key=lambda x: x[1], reverse=True)

        ground_truth_rank = [i for i, (doc, _) in enumerate(ranked_docs) if doc in relevant_docs]
        if ground_truth_rank:
            retrieved_ranks.append(ground_truth_rank[0])

    mrr = sum(1.0 / (rank + 1) for rank in retrieved_ranks) / len(retrieved_ranks)
    return mrr

# # comment
# openai_small_mrr = evaluate_retrieval(msmarco_subset, openai_small)
# openai_large_mrr = evaluate_retrieval(msmarco_subset, openai_large)

# print(f"OpenAI Small Model MRR: {openai_small_mrr:.4f}")
# print(f"OpenAI Large Model MRR: {openai_large_mrr:.4f}")

# hf_mini_mrr = evaluate_retrieval(msmarco_subset, hf_mini)
# hf_mpnet_mrr = evaluate_retrieval(msmarco_subset, hf_mpnet)

# print(f"OpenAI Small Model MRR: {hf_mini_mrr:.4f}")
# print(f"OpenAI Large Model MRR: {hf_mpnet_mrr:.4f}")


In [13]:
# from clearml import Task 
# import torch
# task = Task.init(project_name="Embedding model evaluation", task_name="Compare Embedding Models - Retrieval3") 
# task.execute_remotely(queue_name="default", exit_process=True)

# task.connect({
#     "dataset": "MSMARCO",
#     "num_samples": len(msmarco_subset),
# })
# print("ClearML initialised")

# if torch.cuda.is_available():
#     print("✅ GPU is available! Using:", torch.cuda.get_device_name(0))
# else:
#     print("❌ GPU not detected, running on CPU")


# print('Evaluating openAI models')
# openai_small_mrr = evaluate_retrieval(msmarco_subset, openai_small)
# openai_large_mrr = evaluate_retrieval(msmarco_subset, openai_large)
# print('Evaluating hf models')
# hf_mini_mrr = evaluate_retrieval(msmarco_subset, hf_mini)
# hf_mpnet_mrr = evaluate_retrieval(msmarco_subset, hf_mpnet)

# task.get_logger().report_scalar("Retrieval MRR", "OpenAI Small Model", openai_small_mrr, iteration=1)
# task.get_logger().report_scalar("Retrieval MRR", "OpenAI Large Model", openai_large_mrr, iteration=1)
# task.get_logger().report_scalar("Retrieval MRR", "HF mini Model", hf_mini_mrr, iteration=1)
# task.get_logger().report_scalar("Retrieval MRR", "HF mpnet Model", hf_mpnet_mrr, iteration=1)

# task.close()

# print(f"OpenAI Small Model MRR: {openai_small_mrr:.4f}")
# print(f"OpenAI Large Model MRR: {openai_large_mrr:.4f}")
# print(f"OpenAI Small Model MRR: {hf_mini_mrr:.4f}")
# print(f"OpenAI Large Model MRR: {hf_mpnet_mrr:.4f}")

In [ ]:
# import sys
# print(sys.executable)


c:\Users\guoji\Group-3\gjwenv\Scripts\python.exe


Clustering

In [22]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE 
from clearml import Task

# 初始化 ClearML 任务
task = Task.init(project_name="Embedding model evaluation", task_name="Compare Embedding Models - Clustering")


# 假设 openai_small, openai_large, hf_mini, hf_mpnet 四个函数已经按照之前代码定义好

# 样本文本数据（可以替换为实际的数据集）
texts = [
    "This is a sample sentence about AI.",
    "Artificial intelligence and machine learning are fascinating.",
    "The cat sat on the mat.",
    "Dogs are loyal and friendly.",
    "The economy is showing signs of recovery.",
    "Stock markets have been volatile.",
    "I love to eat pizza.",
    "Italian cuisine is famous for its pasta.",
    "The weather is sunny and warm.",
    "It is raining heavily today."
]

# 聚类数量 (Number of clusters)
n_clusters = 3

# 定义评估函数：计算embedding、进行KMeans聚类并返回Silhouette Score
def evaluate_model(embedding_func, texts, n_clusters):
    # 计算每个文本的embedding
    embeddings = [embedding_func(text) for text in texts]
    embeddings = np.array(embeddings)
    
    # 使用KMeans进行聚类 (Clustering with KMeans)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(embeddings)
    
    # 计算Silhouette Score (Clustering quality metric)
    score = silhouette_score(embeddings, labels)
    return score, labels, embeddings

# 使用每个模型进行评估
results = {}
models = {
    "openai_small": openai_small,
    "openai_large": openai_large,
    "hf_mini": hf_mini,
    "hf_mpnet": hf_mpnet
}

for model_name, func in models.items():
    score, labels, embeddings = evaluate_model(func, texts, n_clusters)
    results[model_name] = {
        "score": score,
        "labels": labels,
        "embeddings": embeddings
    }
    print(f"Model: {model_name}, Silhouette Score: {score}")
    task.get_logger().report_scalar("Clustering silhouette_score", model_name, score, iteration=1)
 
task.close()


ClearML Task: overwriting (reusing) task id=065ab02458004ef297534781693515a4
ClearML results page: https://app.5ccsagap.er.kcl.ac.uk/projects/8f37d229facb4d049719ecba5f506163/experiments/065ab02458004ef297534781693515a4/output/log
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
Model: openai_small, Silhouette Score: 0.049268271395368136
Model: openai_large, Silhouette Score: 0.03291990787330312
Model: hf_mini, Silhouette Score: 0.06918784231168455
Model: hf_mpnet, Silhouette Score: 0.04196056310968392
